<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/18_classification/18_5_2_Bagging_and_Random_Forests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bagging and Random Forests: The Wisdom of Crowds

**Attribution**: Exercises adapted from *Introduction to Modern Statistics (2e)* Chapter 9.6.
Source: [OpenIntro IMS](https://openintro-ims.netlify.app/model-logistic)

**License**: This work is a derivative of [Introduction to Modern Statistics (2e)](https://openintro-ims.netlify.app/) by Mine Çetinkaya-Rundel and Johanna Hardin, used under a [Creative Commons Attribution-ShareAlike 3.0 Unported (CC BY-SA 3.0 US)](https://creativecommons.org/licenses/by-sa/3.0/us/) license.

---

## Introduction
In the previous notebook, we saw that a single decision tree is prone to **overfitting**. Small changes in the data can result in vastly different tree structures. In this notebook, we solve this instability using **Ensemble Methods**.

We will learn that combining many independent (and sometimes noisy) trees creates a model that is remarkably stable and accurate.

### Learning Objectives
By the end of this notebook, you will be able to:
1. **Implement** Bagging to reduce model variance.
2. **Calculate** and interpret Out-of-Bag (OOB) error.
3. **Apply** the Random Forest algorithm to decorrelate individual trees.
4. **Analyze** feature importance to see which markers drive malignancy predictions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score
import os

# Set style
sns.set_context("talk")
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Load data
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

---
## 1. Bagging: Fixing Instability

**Bagging** (Bootstrap Aggregating) works by building 500+ deep trees, each trained on a slightly different "bootstrap sample" of the data. Since each tree is noisy in its own way, the **average** of their predictions is stable.

### Implementation: Bagging with Decision Trees
We will build an ensemble of 500 deep decision trees.

In [ ]:
# Initialize and fit Bagging model
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(), 
    n_estimators=500, 
    oob_score=True, 
    random_state=42
)
bag.fit(X_train, y_train)

print(f"Bagging Accuracy (Test): {accuracy_score(y_test, bag.predict(X_test)):.3f}")
print(f"Out-of-Bag (OOB) Error Score: {bag.oob_score_:.3f}")

### The Out-of-Bag (OOB) Bonus
Notice the `oob_score`. Because we use bootstrap sampling, about **1/3** of our data is left out of each tree. We can treat these "left-out" rows as a test set *during* training. This gives us a validation accuracy estimate **without** needing a separate cross-validation step.

---
## 2. Random Forests: The Improvement

Bagging is great, but it has a problem: if one feature (like `mean concave points`) is very dominant, it will appear at the top of every tree. This makes our trees too similar (correlated).

**Random Forests** fix this by only allowing each tree to look at a **random subset** of features at each split. This forces trees to explore other biological markers and creates a more diverse "crowd."

### Student Task: Fit a Random Forest

In [ ]:
# Initialize and fit Random Forest
rf = RandomForestClassifier(n_estimators=500, random_state=42)
rf.fit(X_train, y_train)

rf_test_acc = accuracy_score(y_test, rf.predict(X_test))
print(f"Random Forest Accuracy (Test): {rf_test_acc:.3f}")

### 3. Feature Importance: What Matters Most?
One of the best ways to interpret an ensemble is to see which features the forest used most effectively to classify malignancy.

In [ ]:
# Plot feature importance
importances = pd.Series(rf.feature_importances_, index=data.feature_names)
top_10 = importances.sort_values(ascending=False).head(10)

top_10.plot(kind='barh')
plt.title("Top 10 Important Features for Breast Cancer Detection")
plt.gca().invert_yaxis()
plt.xlabel("Relative Importance Score")
plt.show()

### Discussion: Prediction Accuracy
Random Forests typically outperform single decision trees and even basic Bagging because of the **decorrelation** of individual trees. 

Next, we will see the third and final logic for ensembling: **Boosting**, where we build trees sequentially to fix each other's mistakes.